# 使用 Bedrock AgentCore Code Interpreter 的 Strands Agents

本实验演示如何将 Strands Agents 与 Amazon Bedrock AgentCore Code Interpreter 集成，以创建能够动态执行代码的 AI 代理。

## 概述

在本实验中，您将：
- 了解 Bedrock AgentCore Code Interpreter 的功能
- 使用 Strands Agents 测试默认的 Code Interpreter
- 创建具有公共网络访问权限的自定义 Code Interpreter
- 比较不同执行环境及其限制

## 前提条件

在开始本实验之前，请确保您已具备以下条件：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 Strands Agents 和 Bedrock AgentCore Python SDK 所需的包：

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 什么是 Bedrock AgentCore Code Interpreter？

Amazon Bedrock AgentCore Code Interpreter 是一个强大的工具，允许 AI 代理在安全的沙箱环境中动态执行代码。主要优势包括：

- **安全执行**：在隔离的沙箱环境中运行 Python 代码
- **动态问题解决**：使代理能够执行计算、分析数据和生成可视化
- **灵活配置**：支持默认沙箱环境和自定义网络启用环境
- **集成就绪**：与 Strands Agents 和其他 AI 框架无缝集成

Code Interpreter 为代理提供了解决需要计算分析、数据处理或数学计算的复杂问题的能力。

### 使用默认 Code Interpreter 测试 Strands Agent

让我们首先使用默认的 AgentCore Code Interpreter 测试 Strands Agent。我们将演示它如何在沙箱环境中生成和执行代码来解决数学问题。

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create a code-gen assistant agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""你是一个通过代码执行来验证所有答案的有用 AI 助手。""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

agent("半径为 8.26 厘米的圆的面积是多少？")

让我们查看代理循环的详细执行流程，以了解代理如何处理请求并生成响应：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

### 了解沙箱环境的限制

默认的 Code Interpreter 在**具有网络隔离的沙箱环境**中运行。这是一项重要的安全功能，可在确保代码执行安全的同时防止未经授权的网络访问。

## 创建具有网络访问权限的自定义 Code Interpreter

为了启用基于 Web 的操作，我们将创建一个具有公共网络访问权限的自定义 Code Interpreter。这展示了 AgentCore 平台针对不同用例的灵活性。

### 步骤 1：初始化 AgentCore 客户端

首先，让我们设置控制平面和数据平面操作所需的客户端：

In [ ]:
from bedrock_agentcore._utils import endpoints
import boto3
import json

region = boto3.session.Session().region_name

data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

cp_client = boto3.client("bedrock-agentcore-control", 
                        region_name=region,
                        endpoint_url=control_plane_endpoint)

dp_client = boto3.client("bedrock-agentcore", 
                        region_name=region,
                        endpoint_url=data_plane_endpoint)

### 步骤 2：创建自定义 Code Interpreter

创建具有公共网络访问权限的自定义 Code Interpreter：

In [ ]:
from botocore.exceptions import ClientError

interpreter_name = "SampleCodeInterpreter"

# Create code interpreter
try:
    interpreter_response = cp_client.create_code_interpreter(
        name=interpreter_name,
        description="Environment for Code Interpreter sample test",
        #executionRoleArn=iam_role_arn, #Required only if the code interpreter need to access AWS resources
        networkConfiguration={
            'networkMode': 'PUBLIC'
        }
    )
    interpreter_id = interpreter_response["codeInterpreterId"]
    print(f"Created interpreter: {interpreter_id}")
except ClientError as e:
    print(f"ERROR: {e}")
    if "already exists" in str(e):
        # If code interpreter already exists, retrieve its ID
        for items in cp_client.list_code_interpreters()['codeInterpreterSummaries']:
            if items['name'] == interpreter_name:
                interpreter_id = items['codeInterpreterId']
                print(f"Code Interpreter ID: {interpreter_id}")
                break
except Exception as e:
    # Show any errors during code interpreter creation
    print(f"ERROR: {e}")

### 步骤 3：创建 Code Interpreter 会话

在自定义 Code Interpreter 中创建一个代码解释器会话：

In [ ]:
from botocore.exceptions import ClientError

session_name = "SampleCodeInterpreterSession"

# Create code interpreter session
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id,
    name=session_name,
    sessionTimeoutSeconds=900
)
session_id = session_response["sessionId"]
print(f"Created session: {session_id}")

### 步骤 4：测试基本功能

让我们验证代码解释器会话是否可以访问互联网，方法是使用 pip 安装 Yahoo Finance Python 包，并使用该包获取今天的 Amazon 股票价格：

In [ ]:
response = dp_client.invoke_code_interpreter(
    codeInterpreterIdentifier=interpreter_id,
    sessionId=session_id,
    name="executeCommand",
    arguments={
        'command': "pip install yfinance"
    }
)
response = dp_client.invoke_code_interpreter(
    codeInterpreterIdentifier=interpreter_id,
    sessionId=session_id,
    name="executeCode",
    arguments={"code": """
        import yfinance as yf
               
        amzn = yf.Ticker('AMZN')
        data = amzn.history(period='1d')
        today_close = data['Close'][-1]
        print(today_close)
        """,
        "language": "python",
        "clearContext": False
    }
)
for event in response["stream"]:
    print(json.dumps(event["result"], indent=2))

## 将自定义 Code Interpreter 与 Strands Agent 配合使用

现在我们将把具有互联网访问权限的自定义 Code Interpreter 集成到 Strands Agent 中。我们将创建共享同一代码解释器会话的自定义 `execute_python` 和 `execute_command` 工具，以增强功能。

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
import boto3

#Reuse the Core Interpreter and Session created above
ci_client = CodeInterpreter(region=boto3.session.Session().region_name)
ci_client.start(identifier=interpreter_id) #initializes a new code interpreter session

@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""
    
    print(f"\n Generated Code: {code}")
    response = ci_client.invoke("executeCode", {
        "code": code,
        "language": "python",
        "clearContext": False
    })
        
    for event in response["stream"]:
        return json.dumps(event["result"])

@tool
def execute_command(command: str, description: str = "") -> str:
    """Execute command in the sandbox."""
    
    print(f"\n Generated Command: {command}")
    response = ci_client.invoke("executeCommand", {
        "command": command
    })
    
    for event in response["stream"]:
        return json.dumps(event["result"])
    

# Create a code gen agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""你是一个通过代码执行来验证所有答案的有用 AI 助手。
                     如果你没有可用的工具来执行任务，你必须生成并执行代码来继续。
                     如果需要，请执行 pip install 来下载所需的包。
                  """,
    tools=[execute_python, execute_command],
)

agent("亚马逊今天的股价是多少？")

ci_client.stop() #stop the current code interpreter session

让我们查看代理循环的详细执行流程，以了解代理如何处理请求并生成响应：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

## 资源清理（可选）

清理我们创建的 AgentCore Runtime 资源，以避免不必要的费用：

In [ ]:
import boto3

cp_client = boto3.client("bedrock-agentcore-control", region_name=region, endpoint_url=control_plane_endpoint)
dp_client = boto3.client("bedrock-agentcore", region_name=region, endpoint_url=data_plane_endpoint)

try:
    print("Cleaning up session and interpreter...")
    dp_client.stop_code_interpreter_session(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id
    )
    print("✓ Session stopped successfully")

    cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
    print("✓ Interpreter deleted successfully")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

## 总结

在本实验中，您成功完成了以下内容：

- ✅ 测试了默认的 Bedrock AgentCore Code Interpreter 功能
- ✅ 创建了具有网络访问功能的自定义 Code Interpreter
- ✅ 将 Code Interpreter 与 Strands Agents 集成以实现动态 Python 执行
- ✅ 执行了包括数据分析、Web 请求和文件操作在内的复杂任务

## AgentCore Code Interpreter 的主要优势

- **动态代码执行**：在 AI 代理工作流中按需运行 Python 代码
- **安全沙箱环境**：具有可配置网络访问的隔离执行环境
- **会话状态管理**：在多次执行之间维护变量和上下文
- **无缝代理集成**：作为 Strands Agents 中的原生工具运行
- **灵活配置**：为特定用例和需求自定义解释器